<a href="https://colab.research.google.com/github/vashirij/wildfire-tinyml-self-sufficiency/blob/main/Notebooks/02_Baseline_Detection_Clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 Baseline Wildfire Detection

This notebook loads the Week 1 simulated environmental dataset and evaluates four baseline wildfire detectors:

1. Rule-based detector  
2. Logistic Regression  
3. Random Forest  
4. Isolation Forest  

It saves metrics, predictions, trained models, confusion matrices, feature importance, latency results, and detection-delay results.


In [ ]:
from pathlib import Path
import random
import time

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Libraries imported successfully.")


## 1. Mount Google Drive and define paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/WildfireProject")

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "simulated"
    / "baseline"
    / "week1_environment_stream.csv"
)

MODEL_DIR = PROJECT_ROOT / "models" / "baseline"
METRICS_DIR = PROJECT_ROOT / "results" / "metrics"
PREDICTIONS_DIR = PROJECT_ROOT / "results" / "predictions"
FIGURES_DIR = PROJECT_ROOT / "figures" / "week1"

for folder in [MODEL_DIR, METRICS_DIR, PREDICTIONS_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())


## 2. Load and inspect the dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Run Notebook 01 and save the generated CSV to Google Drive."
    )

df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
print("Wildfire class counts:")
print(df["wildfire"].value_counts())

print("\nWildfire class proportions:")
print(df["wildfire"].value_counts(normalize=True).round(4))

print("\nMissing values:")
print(df.isna().sum())


## 3. Define features and create a chronological split

In [ ]:
FEATURE_COLUMNS = [
    "temperature_c",
    "humidity_percent",
    "smoke_ppm",
    "co_ppm",
    "wind_speed_kmh",
    "solar_power_w",
    "battery_voltage"
]

TARGET_COLUMN = "wildfire"

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].astype(int).copy()

split_index = int(len(df) * 0.75)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

test_df = df.iloc[split_index:].copy()

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("\nTraining labels:")
print(y_train.value_counts())
print("\nTesting labels:")
print(y_test.value_counts())

if y_train.nunique() < 2:
    raise ValueError("Training data must contain both normal and wildfire observations.")

if y_test.nunique() < 2:
    raise ValueError(
        "Testing data must contain both normal and wildfire observations. "
        "Regenerate the simulated dataset or change the split."
    )


## 4. Evaluation helpers

In [ ]:
def calculate_false_alarm_rate(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    return fp / (fp + tn) if (fp + tn) > 0 else 0.0


def evaluate_predictions(model_name, y_true, y_pred, y_score=None):
    result = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "false_alarm_rate": calculate_false_alarm_rate(y_true, y_pred),
        "roc_auc": np.nan
    }

    if y_score is not None and len(np.unique(y_true)) == 2:
        result["roc_auc"] = roc_auc_score(y_true, y_score)

    return result


## 5. Rule-based detector

In [ ]:
def rule_based_detector(row):
    indicators = 0

    if row["temperature_c"] >= 42:
        indicators += 1
    if row["humidity_percent"] <= 30:
        indicators += 1
    if row["smoke_ppm"] >= 25:
        indicators += 1
    if row["co_ppm"] >= 12:
        indicators += 1

    return int(indicators >= 2)


rule_predictions = X_test.apply(rule_based_detector, axis=1).to_numpy()

rule_metrics = evaluate_predictions(
    "Rule-Based",
    y_test,
    rule_predictions
)

print(rule_metrics)


## 6. Logistic Regression

In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=SEED
        )
    )
])

logistic_model.fit(X_train, y_train)

logistic_predictions = logistic_model.predict(X_test)
logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]

logistic_metrics = evaluate_predictions(
    "Logistic Regression",
    y_test,
    logistic_predictions,
    logistic_probabilities
)

print(logistic_metrics)


## 7. Random Forest

In [ ]:
random_forest_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1
)

random_forest_model.fit(X_train, y_train)

rf_predictions = random_forest_model.predict(X_test)
rf_probabilities = random_forest_model.predict_proba(X_test)[:, 1]

rf_metrics = evaluate_predictions(
    "Random Forest",
    y_test,
    rf_predictions,
    rf_probabilities
)

print(rf_metrics)


## 8. Isolation Forest

In [ ]:
normal_training_data = X_train[y_train == 0].copy()

contamination_value = float(np.clip(y_train.mean(), 0.01, 0.20))

isolation_forest_model = IsolationForest(
    n_estimators=150,
    contamination=contamination_value,
    random_state=SEED,
    n_jobs=-1
)

isolation_forest_model.fit(normal_training_data)

isolation_raw = isolation_forest_model.predict(X_test)
isolation_predictions = np.where(isolation_raw == -1, 1, 0)
isolation_scores = -isolation_forest_model.decision_function(X_test)

isolation_metrics = evaluate_predictions(
    "Isolation Forest",
    y_test,
    isolation_predictions,
    isolation_scores
)

print(isolation_metrics)


## 9. Compare models

In [ ]:
results_df = pd.DataFrame([
    rule_metrics,
    logistic_metrics,
    rf_metrics,
    isolation_metrics
])

results_df = results_df[
    [
        "model",
        "accuracy",
        "precision",
        "recall",
        "f1_score",
        "false_alarm_rate",
        "roc_auc"
    ]
]

display(results_df.round(4))

comparison_path = METRICS_DIR / "week1_model_comparison.csv"
results_df.to_csv(comparison_path, index=False)

print("Saved:", comparison_path)


## 10. Classification reports

In [ ]:
prediction_sets = {
    "Rule-Based": rule_predictions,
    "Logistic Regression": logistic_predictions,
    "Random Forest": rf_predictions,
    "Isolation Forest": isolation_predictions
}

for model_name, predictions in prediction_sets.items():
    print("=" * 70)
    print(model_name)
    print("=" * 70)
    print(
        classification_report(
            y_test,
            predictions,
            target_names=["Normal", "Wildfire"],
            zero_division=0
        )
    )


## 11. Confusion matrices

In [ ]:
for model_name, predictions in prediction_sets.items():
    fig, ax = plt.subplots(figsize=(5, 4))

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        predictions,
        display_labels=["Normal", "Wildfire"],
        values_format="d",
        ax=ax
    )

    ax.set_title(f"{model_name} Confusion Matrix")
    fig.tight_layout()

    filename = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    output_path = FIGURES_DIR / f"{filename}_confusion_matrix.png"
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


## 12. Performance comparison chart

In [ ]:
plot_data = results_df.set_index("model")[
    ["precision", "recall", "f1_score", "false_alarm_rate"]
]

ax = plot_data.plot(kind="bar", figsize=(11, 5))
ax.set_ylabel("Score")
ax.set_title("Baseline Wildfire Detection Performance")
ax.set_ylim(0, 1.05)

plt.xticks(rotation=15)
plt.tight_layout()

comparison_figure_path = FIGURES_DIR / "baseline_model_comparison.png"
plt.savefig(comparison_figure_path, dpi=300, bbox_inches="tight")
plt.show()


## 13. Inference latency

In [ ]:
def measure_latency_ms(predict_function, samples, maximum_samples=500):
    latencies = []
    number_to_test = min(maximum_samples, len(samples))

    for index in range(number_to_test):
        sample = samples.iloc[[index]]

        start = time.perf_counter()
        predict_function(sample)
        end = time.perf_counter()

        latencies.append((end - start) * 1000)

    return {
        "average_latency_ms": np.mean(latencies),
        "median_latency_ms": np.median(latencies),
        "maximum_latency_ms": np.max(latencies)
    }


latency_results = []

for model_name, model in [
    ("Logistic Regression", logistic_model),
    ("Random Forest", random_forest_model),
    ("Isolation Forest", isolation_forest_model)
]:
    result = measure_latency_ms(model.predict, X_test)
    result["model"] = model_name
    latency_results.append(result)

latency_df = pd.DataFrame(latency_results)[
    [
        "model",
        "average_latency_ms",
        "median_latency_ms",
        "maximum_latency_ms"
    ]
]

display(latency_df.round(5))

latency_path = METRICS_DIR / "week1_inference_latency.csv"
latency_df.to_csv(latency_path, index=False)

print("Saved:", latency_path)


These latency values were measured in Google Colab. They are not microcontroller latency measurements.

## 14. Event-level detection delay

In [ ]:
def calculate_event_detection_delays(
    event_ids,
    predictions,
    sampling_interval_seconds=1
):
    evaluation_df = pd.DataFrame({
        "event_id": event_ids.to_numpy(),
        "prediction": np.asarray(predictions)
    })

    fire_event_ids = sorted(
        event_id
        for event_id in evaluation_df["event_id"].unique()
        if event_id > 0
    )

    records = []

    for event_id in fire_event_ids:
        event_rows = evaluation_df[
            evaluation_df["event_id"] == event_id
        ]

        detection_positions = np.where(
            event_rows["prediction"].to_numpy() == 1
        )[0]

        detected = len(detection_positions) > 0

        delay_seconds = (
            int(detection_positions[0]) * sampling_interval_seconds
            if detected
            else np.nan
        )

        records.append({
            "fire_event_id": int(event_id),
            "detected": detected,
            "detection_delay_seconds": delay_seconds
        })

    return pd.DataFrame(records)


delay_frames = []

for model_name, predictions in prediction_sets.items():
    delays = calculate_event_detection_delays(
        event_ids=test_df["fire_event_id"],
        predictions=predictions,
        sampling_interval_seconds=1
    )

    delays["model"] = model_name
    delay_frames.append(delays)

detection_delay_df = pd.concat(delay_frames, ignore_index=True)

display(detection_delay_df)


In [ ]:
delay_summary_df = (
    detection_delay_df
    .groupby("model")
    .agg(
        total_events=("fire_event_id", "count"),
        detected_events=("detected", "sum"),
        average_delay_seconds=("detection_delay_seconds", "mean"),
        maximum_delay_seconds=("detection_delay_seconds", "max")
    )
    .reset_index()
)

delay_summary_df["event_detection_rate"] = (
    delay_summary_df["detected_events"]
    / delay_summary_df["total_events"]
)

display(delay_summary_df.round(4))

delay_path = METRICS_DIR / "week1_detection_delay.csv"
delay_summary_path = METRICS_DIR / "week1_detection_delay_summary.csv"

detection_delay_df.to_csv(delay_path, index=False)
delay_summary_df.to_csv(delay_summary_path, index=False)

print("Saved:", delay_path)
print("Saved:", delay_summary_path)


## 15. Save predictions

In [ ]:
predictions_df = test_df[
    [
        "timestamp",
        "sample_index",
        "fire_event_id",
        "wildfire"
    ]
].copy()

predictions_df["rule_prediction"] = rule_predictions
predictions_df["logistic_prediction"] = logistic_predictions
predictions_df["logistic_probability"] = logistic_probabilities
predictions_df["random_forest_prediction"] = rf_predictions
predictions_df["random_forest_probability"] = rf_probabilities
predictions_df["isolation_forest_prediction"] = isolation_predictions
predictions_df["isolation_anomaly_score"] = isolation_scores

predictions_path = PREDICTIONS_DIR / "week1_baseline_predictions.csv"
predictions_df.to_csv(predictions_path, index=False)

print("Saved:", predictions_path)


## 16. Save trained models

In [ ]:
logistic_path = MODEL_DIR / "logistic_regression.joblib"
rf_path = MODEL_DIR / "random_forest.joblib"
isolation_path = MODEL_DIR / "isolation_forest.joblib"

joblib.dump(logistic_model, logistic_path)
joblib.dump(random_forest_model, rf_path)
joblib.dump(isolation_forest_model, isolation_path)

print("Saved:", logistic_path)
print("Saved:", rf_path)
print("Saved:", isolation_path)


## 17. Random Forest feature importance

In [ ]:
feature_importance_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance": random_forest_model.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance_df)

ax = feature_importance_df.plot(
    x="feature",
    y="importance",
    kind="bar",
    legend=False,
    figsize=(9, 4)
)

ax.set_ylabel("Importance")
ax.set_title("Random Forest Feature Importance")

plt.xticks(rotation=30)
plt.tight_layout()

importance_path = FIGURES_DIR / "random_forest_feature_importance.png"
plt.savefig(importance_path, dpi=300, bbox_inches="tight")
plt.show()


## 18. Final validation

In [ ]:
assert len(rule_predictions) == len(y_test)
assert len(logistic_predictions) == len(y_test)
assert len(rf_predictions) == len(y_test)
assert len(isolation_predictions) == len(y_test)

assert results_df["model"].nunique() == 4

assert comparison_path.exists()
assert predictions_path.exists()
assert logistic_path.exists()
assert rf_path.exists()
assert isolation_path.exists()

best_row = results_df.loc[results_df["f1_score"].idxmax()]

print("=" * 65)
print("BASELINE DETECTION COMPLETED SUCCESSFULLY")
print("=" * 65)
print("Best model by F1-score:", best_row["model"])
print("Best F1-score:", round(best_row["f1_score"], 4))
print("Metrics folder:", METRICS_DIR)
print("Predictions folder:", PREDICTIONS_DIR)
print("Models folder:", MODEL_DIR)
print("Figures folder:", FIGURES_DIR)
